In [0]:
%sql
--- create new schema
create schema if not exists dev.etl
comment 'schema for DLT introduction'

In [0]:
%sql
-- CLONE SAMPLE TABLES
CREATE TABLE IF NOT EXISTS dev.etl.orders_RAW DEEP CLONE samples.tpch.orders;
CREATE TABLE IF NOT EXISTS dev.etl.customer_RAW DEEP CLONE samples.tpch.customer;


In [0]:
# DLT works with t✨💬pes of Datasets
# Streaming Tables (Permanent/Temporary) - Used as Append Data Sources, Incremental data
# Materialized Views - Used for transformations, aggregations or computations
# Views - Used for intermediate Tranformations, not stored in Target Schema

import dlt

In [0]:
# create a streaming table for orders

@dlt.table(
    table_properties={"quality": "bronze"},
    comment = "This is a streaming table for orders"
)
def orders_bronze():
    df = spark.readStream.table("dev.bronze.orders_raw")
    return df

In [0]:
# Create a Materialized View for Customers
@dlt.table(
    table_properties={"quality": "bronze"},
    comment = "This is a streaming table for Customer",
    name = "customer_bronze"
)
def cust_bronze():
    df = spark.read.table("dev.bronze.customer_raw")
    return df

In [0]:
# Create a View to join order with customers
@dlt.view(
    comment = "joined view"
)
def joined_vw():
    df_c = spark.read.table("LIVE.customer_bronze")
    df_o = spark.read.table("LIVE.orders_bronze")

    df_join = df_o.join(df_c, how = "left_outer",on=df_c.c_custkey==df_o.o_custkey)
    return df_join

In [0]:
# Create MV to add new column
from pyspark.sql.functions import current_timestamp
@dlt.table(
    table_properties={"quality": "silver"},
    comment = "Joined table",
    name = "joined_silver"
)
def joined_silver():
    df = spark.read.table("LIVE.joined_vw").withColumn("__insert_date",current_timestamp())
    return df

In [0]:
# Aggregate based on c_mktsegment and find the count of order (c_orderkey)
@dlt.table(
    table_properties = {"quality": "gold"},
    comment = "orders aggregated table"
)
def orders_agg_gold():
    df = spark.read.table("LIVE.joined_silver")

    df_final = df.groupBy("c_mktsegment").agg(count("c_orderkey").alias("sum_orders")).withColumn("__insert_date", current_timestamp())

    return df_final
